## In this notebook, we perform hyperparameter tuning on four SVM models (RBF and Linear kernels) using OneVsOne and OneVsRest classifiers. We use GridSearchCV to find optimal parameters for each model, aiming to enhance classification accuracy and achieve a better balance between bias and variance.


### Param Grid used in the Hyperparam tuning : 
- #### ***estimator__C***: Controls how much the model tries to fit the training data. A higher C means less regularization, while a lower C means more regularization to prevent overfitting.
- #### ***estimator__gamma***: Defines the influence range of each training example in the RBF kernel. Higher gamma values make decision boundaries more complex, while lower ones make them smoother. 


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [2]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [3]:
def calculate_bias_variance(model, X_test, y_test):
    predictions = model.predict(X_test)

    mean_prediction = np.mean(predictions)
    true_mean = np.mean(y_test)
    bias = mean_prediction - true_mean

    variance = np.mean((predictions - mean_prediction) ** 2)

    print(f"Bias: {bias}")
    print(f"Variance: {variance}")

    bias_threshold = 0.15
    variance_threshold = 0.2

    if abs(bias) > bias_threshold and variance < variance_threshold:
        print("Model Status: Underfitting - High bias and low variance indicate the model is too simplistic.")
    elif abs(bias) < bias_threshold and variance > variance_threshold:
        print("Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.")
    elif abs(bias) > bias_threshold and variance > variance_threshold:
        print(
            "Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.")
    else:
        print("Model Status: Balanced - The model has a good balance between bias and variance.")

In [4]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%') 

### The tuned OneVsRest Linear model achieved slightly improved testing accuracy compared to the untuned version (53.776% vs 52.678%). However, both models still show high bias and variance, suggesting they do not generalize well. The improvements were minor, indicating that hyperparameter tuning alone was insufficient to address the model's unbalanced status.

In [5]:
#SVC( OneVsRest Linear )
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

ovr_linear_model = OneVsRestClassifier(SVC(kernel='linear'))

grid_search = GridSearchCV(estimator=ovr_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)


grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 1000}
Best Cross-Validation Accuracy: 0.5360
OneVsRestClassifier with Best Parameters:
Training Accuracy: 54.938%
Testing Accuracy: 53.776%
Bias: -0.35958904109589085
Variance: 0.28554841433664857
Model Status: Unbalanced - Both bias and variance are high, suggesting the model may not generalize well.


### The tuned OneVsRest RBF SVM model showed a reduction in testing accuracy from 59.671% (untuned) to 50.823% (tuned) and an increase in training accuracy to 91.581%. This suggests that hyperparameter tuning failed to improve the model, with persistent overfitting, high variance, and low bias.

In [6]:
#SVC( OneVsRest RBF )
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

ovr_rbf_model = OneVsRestClassifier(SVC(kernel='rbf'))

grid_search = GridSearchCV(estimator=ovr_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)

Fitting 5 folds for each of 35 candidates, totalling 175 fits
Best Parameters: {'estimator__C': 10, 'estimator__gamma': 1}
Best Cross-Validation Accuracy: 0.6141
OneVsRestClassifier RBF with Best Parameters:
Training Accuracy: 91.581%
Testing Accuracy: 50.823%
Bias: -0.08561643835616461
Variance: 0.5770665227997749
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### The tuned OneVsOne Linear model achieved a testing accuracy of 57.136%, which was slightly lower than the untuned model's 57.618%. Both versions continue to show overfitting, with high variance and low bias, indicating that hyperparameter tuning did not significantly enhance the model's generalization ability.

In [7]:
#SVC( OneVsOne Linear )
param_grid = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]  # Values for the regularization parameter C
}

ovo_linear_model = OneVsOneClassifier(SVC(kernel='linear'))

grid_search = GridSearchCV(estimator=ovo_linear_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best Parameters: {'estimator__C': 0.1}
Best Cross-Validation Accuracy: 0.5712
OneVsOneClassifier with Best Parameters:
Training Accuracy: 58.196%
Testing Accuracy: 57.136%
Bias: -0.12328767123287676
Variance: 0.31549071120285227
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### The tuned OneVsOne RBF model experienced a significant drop in testing accuracy, from 59.397% (untuned) to 47.39% (tuned), despite achieving perfect training accuracy of 100.0%. This suggests severe overfitting, with low bias but extremely high variance, indicating that the model fits the training data too well but struggles to generalize.

In [8]:
#SVC( OneVsOne RBF )
param_grid_rbf = {
    'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],  # Regularization parameter
    'estimator__gamma': [0.001, 0.01, 0.1, 1, 10]  # Gamma parameter for the RBF kernel
}

ovo_rbf_model = OneVsOneClassifier(SVC(kernel='rbf'))

grid_search = GridSearchCV(estimator=ovo_rbf_model, param_grid=param_grid_rbf, cv=5, scoring='accuracy', verbose=1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")

cv_results = cross_validate(best_model, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF with Best Parameters:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(best_model, X_test, y_test)

Fitting 5 folds for each of 35 candidates, totalling 175 fits
Best Parameters: {'estimator__C': 1000, 'estimator__gamma': 1}
Best Cross-Validation Accuracy: 0.6132
OneVsOneClassifier RBF with Best Parameters:
Training Accuracy: 100.0%
Testing Accuracy: 47.39%
Bias: -0.04452054794520599
Variance: 0.5609284105835992
Model Status: Overfitting - Low bias and high variance indicate the model fits training data too well.


### Based on the analysis, none of the tuned models significantly improved generalization, and most continued to overfit. The OneVsRest Linear SVM showed moderate improvement, maintaining a stable bias-variance balance, and is recommended for the ensemble model due to better testing accuracy and simpler complexity. The OneVsOne Linear SVM is a close second, with similar testing accuracy and balance.